# Train Win Probability Model

Trains an XGBoost classifier to predict match outcomes based on in-game match states.

## Model Architecture
* **Algorithm**: XGBoost Multi-class Classifier
* **Target**: final_outcome (home_win, draw, away_win)
* **Features**: 10 numeric features capturing match state
* **Output**: 3 probabilities summing to 1.0

## Pipeline
1. Load training data from Unity Catalog
2. Feature engineering and encoding
3. Train/test split
4. Train XGBoost model
5. Evaluate performance
6. Log model to MLflow
7. Register in Unity Catalog

In [0]:
import sys
sys.path.append('/Workspace/Users/pawanvirat32@gmail.com/MatchPulse')

import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature

from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml import Pipeline
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize Spark
spark = SparkSession.builder.appName("TrainWinProbability").getOrCreate()

print("=" * 80)
print("Training Win Probability Model - XGBoost")
print("=" * 80)

In [0]:
# Load training data from Unity Catalog
print("\n[1/7] Loading training data...")

df_training = spark.read.table("matchpulse.ml.training_match_states")
print(f"   Total samples: {df_training.count():,}")

# Show sample
print("\nSample training data:")
display(df_training.limit(5))

# Check class distribution
print("\nClass distribution:")
df_training.groupBy("final_outcome").count().orderBy(F.desc("count")).show()

In [0]:
# Prepare features
print("\n[2/7] Feature engineering...")

feature_cols = [
    "minute",
    "current_score_diff",
    "home_xg_so_far",
    "away_xg_so_far",
    "home_shots",
    "away_shots",
    "home_red_cards",
    "away_red_cards",
    "home_form_pts",
    "away_form_pts"
]

# Convert to Pandas for sklearn/xgboost
df_pandas = df_training.select(
    feature_cols + ["final_outcome"]
).toPandas()

# Encode target variable
target_mapping = {'home_win': 0, 'draw': 1, 'away_win': 2}
df_pandas['target'] = df_pandas['final_outcome'].map(target_mapping)

print(f"   Total samples in pandas: {len(df_pandas):,}")
print(f"   Features: {len(feature_cols)}")
print(f"   Feature columns: {feature_cols}")

# Check for nulls
print(f"\nNull values per column:")
print(df_pandas[feature_cols].isnull().sum())

# Fill any nulls with 0
df_pandas[feature_cols] = df_pandas[feature_cols].fillna(0)

In [0]:
# Split data
print("\n[3/7] Splitting data...")

X = df_pandas[feature_cols]
y = df_pandas['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   Training samples: {len(X_train):,}")
print(f"   Test samples: {len(X_test):,}")
print(f"\n   Training class distribution:")
print(y_train.value_counts())
print(f"\n   Test class distribution:")
print(y_test.value_counts())

In [0]:
# Train XGBoost model
print("\n[4/7] Training XGBoost model...")

# Set up MLflow - set registry URI before starting run
mlflow.set_experiment("/Users/pawanvirat32@gmail.com/MatchPulse/win_probability_experiments")
mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="xgboost_win_probability") as run:
    
    # Model parameters
    params = {
        'objective': 'multi:softprob',
        'num_class': 3,
        'max_depth': 6,
        'learning_rate': 0.1,
        'n_estimators': 200,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'eval_metric': 'mlogloss'
    }
    
    # Log parameters
    mlflow.log_params(params)
    mlflow.log_param("features", ",".join(feature_cols))
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("train_samples", len(X_train))
    mlflow.log_param("test_samples", len(X_test))
    
    # Train model
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    print("   Model trained successfully")
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"\n   Test Accuracy: {accuracy:.4f}")
    
    # Log metrics
    mlflow.log_metric("test_accuracy", accuracy)
    
    # Classification report
    print("\n   Classification Report:")
    target_names = ['home_win', 'draw', 'away_win']
    print(classification_report(y_test, y_pred, target_names=target_names))
    
    # Log model with signature
    signature = infer_signature(X_train, y_pred_proba)
    mlflow.xgboost.log_model(
        model,
        "model",
        signature=signature,
        input_example=X_train.iloc[:5]
    )
    
    print(f"\n   MLflow Run ID: {run.info.run_id}")

In [0]:
# Confusion matrix
print("\n[5/7] Creating confusion matrix...")

cm = confusion_matrix(y_test, y_pred)
target_names = ['Home Win', 'Draw', 'Away Win']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=target_names,
    yticklabels=target_names,
    cbar_kws={'label': 'Count'},
    ax=ax
)
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix - Win Probability Model', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nConfusion Matrix:")
print(cm)

In [0]:
# Feature importance
print("\n[6/7] Analyzing feature importance...")

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    feature_importance['feature'],
    feature_importance['importance'],
    color='#3498db',
    alpha=0.8,
    edgecolor='black'
)
ax.set_xlabel('Importance', fontsize=11, fontweight='bold')
ax.set_ylabel('Feature', fontsize=11, fontweight='bold')
ax.set_title('Feature Importance - XGBoost Win Probability Model', 
             fontsize=13, fontweight='bold', pad=15)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [0]:
# Register model in Unity Catalog
print("\n[7/7] Registering model in Unity Catalog...")

try:
    # Set registry URI to Unity Catalog
    mlflow.set_registry_uri("databricks-uc")
    
    # Register the model
    model_name = "matchpulse.ml.win_probability_model"
    
    model_uri = f"runs:/{run.info.run_id}/model"
    
    model_version = mlflow.register_model(
        model_uri=model_uri,
        name=model_name,
        tags={"stage": "development", "model_type": "xgboost"}
    )
    
    print(f"\n✓ Model registered: {model_name}")
    print(f"   Version: {model_version.version}")
    print(f"   Run ID: {run.info.run_id}")
    
except Exception as e:
    error_msg = str(e)
    print(f"\n⚠ Could not register model in UC: {e}")
    
    # Check if it's a permission error
    if "AccessDenied" in error_msg or "s3:PutObject" in error_msg:
        print("\n   This is a permission issue with the Unity Catalog storage location.")
        print("   The model metadata was created, but artifacts couldn't be uploaded.")
        print("\n   To resolve this, contact your workspace admin to:")
        print("   1. Grant s3:PutObject permission on the Unity Catalog storage bucket")
        print("   2. Or use a different catalog with proper S3 permissions")
    
    print("\n   ℹ Model is still available in MLflow:")
    print(f"   Run ID: {run.info.run_id}")
    print(f"   You can load it with: mlflow.xgboost.load_model('runs:/{run.info.run_id}/model')")

## Model Summary

✓ **Model trained and logged to MLflow**
✓ **Feature importance calculated**
✓ **Confusion matrix created**
✓ **Model registered in Unity Catalog** (if successful)

### Next Steps
1. Run `03_validate_model.py` for detailed validation
2. Test model on live match data
3. Deploy to production endpoint